<div style="padding:20px; border-radius:6px; background-color:#ffe6e6; color:#b30000; border-left:5px solid #b30000;">
<b>Jalankan notebook ini dengan environment <code>mandala_env</code>.</b><br/>
Siapkan <code>VAULT_TOKEN</code> di file <code>.env</code> pada <b>root workdir proyek Anda</b> (atau export di environment).
</div>

# Tutorial: Akses Data INACAWO via Mandala

Marine Data Lake (**Mandala**) menyimpan observasi MMS1 dan dataset model (termasuk INACAWO)
di object storage. Di Mandalahub, Anda membaca data lewat protokol **S3** sebagai
**Zarr / zcollection** — tanpa mengunduh file NetCDF utuh ke workspace.

Notebook ini melanjutkan materi OPeNDAP: fokus living archive, catalog Mandala,
lalu **case study briefing metocean Selat Sunda** yang sama (angin + gelombang + arus)
dengan layout operasional OFS.


## 1. Apa itu Mandala / Mandalahub / Zarr?

| Konsep | Arti singkat |
|---|---|
| **Data lake** | Penyimpanan fleksibel; data masuk tanpa wajib transform penuh dulu |
| **Mandala** | Data lake maritim BMKG (observasi + NWP) di S3 on-prem |
| **Zarr / zcollection** | Format chunked; baca hanya potongan yang dibutuhkan, cocok object storage |
| **Mandalahub** | JupyterHub analisis di jaringan lokal ke S3 Mandala |

Untuk INACAWO di Mandala: arsip **living FULL** cycle **00Z & 12Z** — ditumpuk / di-override
tiap cycle terbaru (bukan snapshot beku per run). Hindcast multi-dekade juga tersedia
lewat catalog `model_hindcast`.

**Bedakan dari OPeNDAP:** OPeNDAP = file NetCDF forecast retensi pendek + subset remote;
Mandala = arsip panjang + I/O paralel Zarr di Mandalahub.


## 2. Struktur data Mandala vs OPeNDAP

Keduanya berasal dari postproc ilmiah INACAWO yang sama, tetapi **kemasan berbeda**.
Selalu cek `.attrs` sebelum membandingkan angka antar sumber.

### Dimensi & naming
| | OPeNDAP | Mandala (contoh) |
|---|---|---|
| Waktu | `time` | `TSTEP` |
| Vertikal | `depth` (Flows) / — | `LAY` (layer; di WRF = level tekanan) |
| Spatials | `lat`, `lon` | `ROW`, `COL` (+ coord `lat`/`lon`) |
| SWH | `hs` | `sig_wave_height` |
| Wind U/V | `uwnd` / `vwnd` | `U_WIND` / `V_WIND` (`inacawo_wrf_3d`) |
| Current | `u` / `v` | `u_sur_eastward` / `v_sur_northward` (atau `ubar_*`) |
| SSH | `zeta` | `zeta` |

### Collection yang dipakai di tutorial
| Path collection | Isi tipikal |
|---|---|
| `mandala-prod1/datalake/model/collections/inacawo_ocean_2d` | ocean/wave 2D surface fields |
| `mandala-prod1/datalake/model/collections/inacawo_wrf_3d` | atmosfer 3D |

### Satuan & arah (ingat!)
- Angin Mandala `U_WIND`/`V_WIND` (`inacawo_wrf_3d`): **m/s** → plot OFS pakai `wind_in_mps=True`.
- Arus Mandala `u_sur_*` / `ubar_*`: **m/s** → plot OFS pakai `current_in_mps=True`
  (beda dari OPeNDAP Flows yang biasanya **cm/s**).
- Gelombang `sig_wave_height`: **m**; `mean_wave_dir` dll.: derajat **meteorological coming from**
  (CF `*_from_direction`). Hyrax OPeNDAP `dir` beda encoding (math *going to*) — lihat tutorial OPeNDAP.
- Komponen `*_eastward` / `*_northward` = vektor *ke arah* (CF).
- Helper `plot_ofs_map(..., wave_dir_convention="meteo_from")` (default) → panah = arah jalar.


## 3. Import & koneksi S3 via Vault

Jangan hard-code Access Key / Secret Key di notebook.
Token Vault bisa di-set di environment atau di cell di bawah (jangan commit token).


In [ ]:
import os
from datetime import datetime
from pathlib import Path
from typing import Literal

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zcollection as zc
import zcollection.fs_utils as zc_fs
import mandala

from training_ofs import (
    find_project_root,
    load_project_env,
    load_s3_options_from_vault,
    plot_ofs_map,
    list_params,
)

# Domain case study — sama dengan opendap-tutorial.ipynb
LAT_SLICE = slice(-7.5, -5.0)
LON_SLICE = slice(104.5, 107.5)
TARGET_TIME = pd.Timestamp("2026-09-12 12:00:00")

# Portable: each participant workdir has its own clone + .env
PROJECT_ROOT = find_project_root()
ENV_PATH = load_project_env(required_keys=("VAULT_TOKEN",))
VAULT_TOKEN = os.environ["VAULT_TOKEN"]

print("Project root:", PROJECT_ROOT)
print("Loaded env:", ENV_PATH if ENV_PATH.is_file() else "(env vars only)")

s3_options = load_s3_options_from_vault(token=VAULT_TOKEN)

fs = fsspec.filesystem(
    "s3",
    key=s3_options["key"],
    secret=s3_options["secret"],
    client_kwargs=s3_options["client_kwargs"],
)
fs


## 4. Catalog Mandala

Catalog memetakan dataset observasi / model / hindcast.
- `obs` → observasi MMS1
- `model` → living forecast archive (termasuk INACAWO operasional)
- `model_hindcast` → hindcast (retensi panjang; hati-hati load tanpa filter)


In [ ]:
def get_catalog(
    database: Literal["obs", "model", "model_hindcast"],
    s3_options: dict,
):
    catalog = {
        "obs": "catalog.json",
        "model": "catalog.json",
        "model_hindcast": "catalog_hindcast.json",
    }.get(database, "catalog.json")
    if database == "model_hindcast":
        database = "model"
    catalog_path = os.path.join("mandala-prod1/datalake/", database, catalog)
    return mandala.MandalaCatalog(
        path=catalog_path,
        protocol="s3",
        options=s3_options,
    )


In [ ]:
catalogobs = get_catalog("obs", s3_options)
catalogobs


In [ ]:
catalogmodel = get_catalog("model", s3_options)
catalogmodel


In [ ]:
catalogmodelhindcast = get_catalog("model_hindcast", s3_options)
catalogmodelhindcast


## 5. Membaca collection INACAWO dengan zcollection

Semua dataset disimpan sebagai **zcollection** (di atas Zarr), efisien diproses dengan `xarray`.

**Wajib:** pakai `filters` tanggal sempit. Load tanpa filter bisa menarik ratusan partition.


In [ ]:
fs.invalidate_cache()
ocean2d_collection = zc.open_collection(
    path="mandala-prod1/datalake/model/collections/inacawo_ocean_2d",
    mode="r",
    filesystem=zc_fs.get_fs(filesystem=fs),
)

# Jaga window kecil agar ramah sistem (sesuaikan jika partition berubah)
date_filter = (
    "(year == 2026 and month == 9 and day >= 12) or "
    "(year == 2026 and month == 9 and day <= 13)"
)

dset = ocean2d_collection.load(delayed=True, filters=date_filter).to_xarray()
dset


### Inspect atribut (latihan)

Bandingkan nama variabel Mandala dengan short name OPeNDAP dari tutorial sebelumnya.


In [ ]:
def summarize_mandala(ds: xr.Dataset) -> pd.DataFrame:
    rows = []
    for name, da in ds.data_vars.items():
        rows.append(
            {
                "variable": name,
                "dims": tuple(da.dims),
                "units": da.attrs.get("units"),
                "long_name": da.attrs.get("long_name") or da.attrs.get("var_desc"),
                "standard_name": da.attrs.get("standard_name"),
            }
        )
    return pd.DataFrame(rows).sort_values("variable")

summarize_mandala(dset).head(20)


In [ ]:
# Quicklook bawaan xarray (bukan layout OFS) — crop Selat Sunda dulu
t_idx_preview = int(np.argmin(np.abs(pd.to_datetime(dset["TSTEP"].values) - TARGET_TIME)))
(
    dset["sig_wave_height"]
    .isel(TSTEP=t_idx_preview, LAY=0)
    .sel(ROW=LAT_SLICE, COL=LON_SLICE)
    .plot(figsize=(6, 4))
)
plt.title("sig_wave_height (m) — quicklook subset Selat Sunda")
plt.show()
print("Preview TSTEP:", pd.to_datetime(dset["TSTEP"].values[t_idx_preview]))


## 6. Hands-on Atmos (WRF 3D) untuk angin

Angin di Mandala diambil dari collection `inacawo_wrf_3d` (`U_WIND` / `V_WIND`, **m/s**).
Level terendah `LAY=0` ≈ 1000 hPa (dekat permukaan) — analog praktis angin Atmos OPeNDAP.

Tetap filter tanggal sempit, sama window dengan ocean 2D.


In [ ]:
fs.invalidate_cache()
wrf3d_collection = zc.open_collection(
    path="mandala-prod1/datalake/model/collections/inacawo_wrf_3d",
    mode="r",
    filesystem=zc_fs.get_fs(filesystem=fs),
)

dset_wrf = wrf3d_collection.load(delayed=True, filters=date_filter).to_xarray()
print(dset_wrf[["U_WIND", "V_WIND"]])
print("U_WIND units:", dset_wrf["U_WIND"].attrs.get("units"))
print("LAY[0] (Pa):", float(dset_wrf["LAY"].values[0]))


## 7. Case study: briefing metocean jalur Selat Sunda

**Skenario.** Sama dengan `opendap-tutorial.ipynb`: forecaster diminta ringkasan singkat
kondisi angin, gelombang, dan arus di jalur Selat Sunda untuk satu waktu forecast.

**Aturan beban:** bbox kecil (`LAT_SLICE`/`LON_SLICE`), 1 waktu, 1–2 variabel per produk —
lalu plot dengan **layout operasional OFS** (logo, title, colorbar, legend arah).


Siapkan slice bersama (waktu + bbox), lalu tiga peta: angin → gelombang → arus.


In [ ]:
# Waktu & bbox — selaras case study OPeNDAP (~t+12 jam dari cycle 00Z 12 Sep 2026)
t_ocean = pd.to_datetime(dset["TSTEP"].values)
t_wrf = pd.to_datetime(dset_wrf["TSTEP"].values)
t_idx = int(np.argmin(np.abs(t_ocean - TARGET_TIME)))
t_idx_wrf = int(np.argmin(np.abs(t_wrf - TARGET_TIME)))
fcst = pd.Timestamp(t_ocean[t_idx]).to_pydatetime()
baserun = datetime(2026, 9, 12, 0, 0)  # cycle referensi (sama konsep OPeNDAP)

atm = (
    dset_wrf[["U_WIND", "V_WIND"]]
    .isel(TSTEP=t_idx_wrf, LAY=0)
    .sel(ROW=LAT_SLICE, COL=LON_SLICE)
)
wav = (
    dset[["sig_wave_height", "mean_wave_dir"]]
    .isel(TSTEP=t_idx, LAY=0)
    .sel(ROW=LAT_SLICE, COL=LON_SLICE)
)
flo = (
    dset[["u_sur_eastward", "v_sur_northward"]]
    .isel(TSTEP=t_idx, LAY=0)
    .sel(ROW=LAT_SLICE, COL=LON_SLICE)
)

print("Forecast time:", fcst)
print("atm shape:", atm["U_WIND"].shape, "| wav:", wav["sig_wave_height"].shape, "| flo:", flo["u_sur_eastward"].shape)
print("current units:", flo["u_sur_eastward"].attrs.get("units"))


In [ ]:
lon_a = atm["COL"].values
lat_a = atm["ROW"].values

fig_ws, ax_ws = plot_ofs_map(
    lon_a,
    lat_a,
    np.sqrt(atm["U_WIND"].values ** 2 + atm["V_WIND"].values ** 2),
    param="ws",
    u=atm["U_WIND"].values,
    v=atm["V_WIND"].values,
    wind_in_mps=True,  # Mandala WRF = m/s
    baserun=baserun,
    forecast=fcst,
    wind_input="INACAWO",
    map_area="Selat Sunda",
    quiver_skip=4,
)
plt.show()

In [ ]:
fig_hs, ax_hs = plot_ofs_map(
    wav["COL"].values,
    wav["ROW"].values,
    wav["sig_wave_height"].values,
    param="swh",
    direction=wav["mean_wave_dir"].values,
    wave_dir_convention="meteo_from",  # CF *_from_direction
    baserun=baserun,
    forecast=fcst,
    wind_input="INACAWO",
    map_area="Selat Sunda",
    quiver_skip=4,
)
plt.show()

In [ ]:
# Mandala u/v permukaan sudah m/s → current_in_mps=True (OPeNDAP Flows biasanya cm/s)
fig_csd, ax_csd = plot_ofs_map(
    flo["COL"].values,
    flo["ROW"].values,
    np.sqrt(flo["u_sur_eastward"].values ** 2 + flo["v_sur_northward"].values ** 2),
    param="csd",
    u=flo["u_sur_eastward"].values,
    v=flo["v_sur_northward"].values,
    current_in_mps=True,
    baserun=baserun,
    forecast=fcst,
    wind_input="INACAWO",
    map_area="Selat Sunda",
    quiver_skip=4,
)
plt.show()

### Interpretasi singkat (isi sendiri saat praktik)

Tulis 3–5 kalimat berdasarkan ketiga peta di atas, misalnya:
- Kekuatan & arah angin dominan di jalur
- Tinggi gelombang signifikan relatif terhadap ambang operasional lokal
- Arus permukaan — apakah mendukung / menentang alur pelayaran

**Catatan arah:** panah mengikuti konvensi helper `ofs-plotter`. Jika membandingkan dengan
sumber lain (termasuk OPeNDAP cycle yang sama), cocokkan dulu apakah arah “from” atau “to”.

## 8. Best practice & catatan lanjutan

**Lakukan**
- Baca langsung S3 + Zarr di Mandalahub (jaringan lokal)
- Filter tanggal / pilih variabel sebelum compute penuh
- Cek `units` & naming sebelum bandingkan dengan OPeNDAP
- Plot case study Selat Sunda (`ws` / `swh` / `csd`) pakai `plot_ofs_map` — sama skenario dengan OPeNDAP
- Cek flag satuan: Mandala angin/arus **m/s** → `wind_in_mps=True`, `current_in_mps=True`

**Hindari**
- Unduh file OPeNDAP penuh lalu upload ke Mandalahub
- Load hindcast multi-tahun tanpa filter
- Hard-code secret S3 di notebook / git

### Tip hindcast `TFLAG`
Pada beberapa collection hindcast, `TFLAG` punya ukuran `VAR` yang berubah antar era.
Jika `load()` gagal dengan *conflicting dimensions*, exclude `TFLAG` via
`selected_variables=[...]` dan tetap filter tanggal sempit.


## 9. Ringkasan jalur akses

| Kebutuhan | Jalur |
|---|---|
| Subset remote cepat / luar Mandalahub | OPeNDAP Hyrax |
| Analisis deret waktu / domain luas di Mandalahub | Mandala S3 + Zarr |
| Plot selaras produk operasional | `training_ofs.plot_ofs_map` (param dari ofs-plotter) |

Parameter plot tersedia: lihat `list_params()` di bawah.


In [ ]:
print("OFS plot params:", ", ".join(list_params()))